# 04 · Feature extraction

| | |
|---|---|
| **入力** | `data/clean/<subject>.csv` |
| **出力** | 窓化した特徴量行列 + フィット済み `StandardScaler`（`.npz` / `.joblib`） |

流れ: EEG の帯域分割 → 時系列ブロック分割 → スライディング窓 → 特徴量 ブランチ → 標準化.

| ブランチ | 窓長 | 特徴量 |
|---|---|---|
| EEG Hjorth | 0.5 s | チャネルごとの activity + mobility, 帯域ごと（α, β） |
| EMG RMS | 0.2 s | 筋ごとの二乗平均平方根 |
| モーション | 0.2 s | マーカー平均位置 |
| 環境 | – | 椅子高, 段差高 |

In [ ]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler

from motion_intent.preprocessing import (
    merge_xz_all, drop_single_axis_marker_cols, append_band_columns,
    bandpass_by_session,
)
from motion_intent.labeling import labels_to_ids
from motion_intent.windowing import (
    split_by_time_block, make_reference_indices, windows_at,
    transition_offset_refs,
)
from motion_intent.features import (
    extract_hjorth_fast, extract_rms, extract_mean_position, build_env_features,
)

## Load & band-split

In [ ]:
subject = 'haru'      # ← 自分の被験者フォルダ名に変更
clean_csv = config.CLEAN_DIR / f'{subject}.csv'
assert clean_csv.exists(), f'{clean_csv} not found - run notebook 03 first'
df = pd.read_csv(clean_csv)

# Drop XDF passthrough columns nothing here uses: EEG_Timestamp/Counter/
# Interpolate/HardwareMarker/Markers are stream metadata, not signal (~90%
# NaN in the merged frame), and RigidBodies_* (chair/stair tracking) is very
# sparse and not part of any feature branch below. Left in, either would make
# the blanket dropna() a few cells down discard almost every row for reasons
# unrelated to the columns actually used.
drop_cols = [
    c for c in df.columns
    if c.startswith('RigidBodies_')
    or c in ('EEG_Timestamp', 'EEG_Interpolate', 'EEG_Counter', 'EEG_HardwareMarker', 'EEG_Markers')
]
df = df.drop(columns=drop_cols)

df = drop_single_axis_marker_cols(merge_xz_all(df))

# EMG can have genuine session-boundary gaps (e.g. the sensor stopped a little
# before the rest of the recording); forward-fill per session rather than let
# dropna() discard otherwise-valid rows for it - consistent with notebook 03
# treating this pipeline as a stand-in for a real-time system.
df[config.EMG_COLS] = df.groupby('Session')[config.EMG_COLS].transform(lambda s: s.ffill().bfill())

df = df.dropna().reset_index(drop=True)

# 必要ならワンホット列から整数ラベルを作る／文字列ラベルなら ID 化する
# (pandas>=2.something can give a "str" dtype for CSV text columns, not the
# legacy "object" - checking dtype == object misses that, so check the
# opposite: is it already an integer dtype?)
if 'label' not in df.columns:
    assert set(config.CLASS_NAMES).issubset(df.columns), "no 'label' column and no one-hot class columns"
    df['label'] = df[config.CLASS_NAMES].to_numpy().argmax(axis=1)
elif not pd.api.types.is_integer_dtype(df['label']):
    df['label'] = labels_to_ids(df['label'], config.CLASS_NAMES)
    assert (df['label'] >= 0).all(), 'label contains a class name outside config.CLASS_NAMES'

# 運動野中心 EEG チャネルの α / β 帯域コピーを追加
df = append_band_columns(df, config.EEG_CH_FEATURE, 'Session', config.FS,
                         bands=config.EEG_BANDS, drop_original=False)
# EMG 帯域通過
df = bandpass_by_session(df, config.EMG_COLS, 'Session', config.FS,
                         band=config.EMG_BAND)

## Chronological split, per session

In [ ]:
splits = {'train': [], 'val': [], 'test': []}
for _, df_sess in df.groupby('Session'):
    tr, va, te = split_by_time_block(df_sess)
    splits['train'].append(tr); splits['val'].append(va); splits['test'].append(te)

## Windowing

In [ ]:
COLS = {
    'eeg_a':   [f'{c}_alpha' for c in config.EEG_CH_FEATURE],
    'eeg_b':   [f'{c}_beta'  for c in config.EEG_CH_FEATURE],
    'emg':     config.EMG_COLS,
    'motion':  config.MOTION_COLS,
}

def window_split(frames):
    out = {k: [] for k in COLS}
    ys = []
    for df_sess in frames:
        t = df_sess['t_sec'].to_numpy()
        ref = make_reference_indices(t, config.WIN_SEC_EEG, config.STEP_SEC)
        ref = ref[ref >= config.WIN_EEG]
        y = df_sess['label'].to_numpy()
        # 0.5 s EEG 窓の中心サンプルのラベルを採用
        ys.append(y[ref - config.WIN_EEG // 2])
        for name, cols in COLS.items():
            win = config.WIN_EEG if name.startswith('eeg') else config.WIN_EMG
            out[name].append(windows_at(df_sess[cols].to_numpy(), ref, win))
    return {k: np.concatenate(v) for k, v in out.items()}, np.concatenate(ys)

Xw, yw = {}, {}
for part in ('train', 'val', 'test'):
    Xw[part], yw[part] = window_split(splits[part])

## Feature branches

EEG 特徴量は α / β 帯域それぞれの Hjorth（activity・mobility）のみ.

In [ ]:
def features_for(part):
    X = Xw[part]
    hjorth = np.concatenate([extract_hjorth_fast(X['eeg_a']),
                             extract_hjorth_fast(X['eeg_b'])], axis=1)
    rms = extract_rms(X['emg'])
    motion = extract_mean_position(X['motion'])
    env = build_env_features(len(rms), config.DEFAULT_CHAIR_HEIGHT_M,
                             config.DEFAULT_STAIR_HEIGHT_M)
    return dict(hjorth=hjorth, rms=rms, motion=motion, env=env)

F = {part: features_for(part) for part in ('train', 'val', 'test')}

## Standardise (fit on train) and save

In [ ]:
scalers = {}
for key in ('hjorth', 'rms', 'motion'):
    sc = StandardScaler().fit(F['train'][key])
    scalers[key] = sc
    for part in ('train', 'val', 'test'):
        F[part][key] = sc.transform(F[part][key])

art_dir = config.DATA_DIR / 'features'
art_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scalers, art_dir / f'{subject}_scalers.joblib')
for part in ('train', 'val', 'test'):
    np.savez(art_dir / f'{subject}_{part}.npz', y=yw[part], **F[part])
print('saved to', art_dir)

## Offset windows around transitions

評価（ノートブック 06）用のデータ. 各ラベル遷移点を候補オンセットとみなし, テスト窓を遷移点に対して −400 ms 〜 +400 ms ずらして再抽出する. ラベルは **遷移後のクラス**（開始しようとしている動作）を付与する. 特徴量は本編と 同じ関数で作り, **学習集合でフィット済みの scaler** をそのまま適用する.

出力: `data/features/<subject>_offset_<ms>.npz`（オフセット 1 つにつき 1 ファイル）.

In [ ]:
offsets_ms = list(range(-config.OFFSET_RANGE_MS, config.OFFSET_RANGE_MS + 1,
                        config.OFFSET_STEP_MS))

def offset_windows(off_samples):
    mods = {k: [] for k in COLS}
    ys = []
    for df_sess in splits['test']:
        lab = df_sess['label'].to_numpy()
        refs, y = transition_offset_refs(lab, off_samples, len(df_sess), config.WIN_EEG)
        ys.append(y)
        for name, cols in COLS.items():
            win = config.WIN_EEG if name.startswith('eeg') else config.WIN_EMG
            mods[name].append(windows_at(df_sess[cols].to_numpy(), refs, win))
    return {k: np.concatenate(v) for k, v in mods.items()}, np.concatenate(ys)

for ms in offsets_ms:
    Xoff, yoff = offset_windows(int(ms * config.FS / 1000))
    hjorth = np.concatenate([extract_hjorth_fast(Xoff['eeg_a']),
                             extract_hjorth_fast(Xoff['eeg_b'])], axis=1)
    feats = {
        'hjorth': scalers['hjorth'].transform(hjorth),
        'rms':    scalers['rms'].transform(extract_rms(Xoff['emg'])),
        'motion': scalers['motion'].transform(extract_mean_position(Xoff['motion'])),
    }
    np.savez(art_dir / f'{subject}_offset_{ms}.npz', y=yoff, **feats)

print(f'wrote {len(offsets_ms)} offset files')